<!-- dads-lab-header -->
<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M13/M13_Lab1_Fine_Tuning_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🎯 M13 Lab 1 — Fine-Tuning & Deployment</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~50 min + fine-tune queue time (~15 min)</p>
</div>

> **📌 Lab note.** Fine-tuning a small dataset with `gpt-4.1-mini` costs roughly **$0.30–1.00**. The fine-tuning job itself takes ~15 min to complete in OpenAI's queue. You can run all cells and inspect the output without actually submitting a job — just set `SUBMIT_JOB = False` in Part 3.

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
import os
import importlib.util
if importlib.util.find_spec('dads5250') is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp, pretty_print, lab_pill, setup_openai,
    DEFAULT_CHAT_MODEL, DEFAULT_MINI_MODEL,
)

lab_pill('M13 Lab 1 — Fine-Tuning & Deployment')
client = setup_openai()


---
## 🔹 Part 1 — The Decision Ladder: When to Fine-Tune

Fine-tuning is powerful and **usually the wrong first answer**. Before reaching for it, climb the ladder:

| Rung | Technique | Cost | When it works |
|------|-----------|------|---------------|
| 1 | **Prompt engineering** | Free | Format, style, behavior tweaks |
| 2 | **Few-shot prompting** | Free | Need consistent examples |
| 3 | **RAG** | Low | Large / changing knowledge |
| 4 | **Tool / function calling** | Low | Live data, calculations, APIs |
| 5 | **Fine-tuning** | Medium–High | Durable style, speed, cost reduction |

**Fine-tuning IS the right answer when:**
- You need a **very specific, consistent style** (brand voice, medical report template)
- You are calling the model **millions of times** and want to bake prompts into weights to save tokens
- A smaller fine-tuned model can **match a larger base model** at fraction of the cost
- You need **sub-100ms latency** and a smaller model is the only option

**Fine-tuning is NOT the answer when:**
- The knowledge changes frequently (use RAG instead)
- You just want it to follow instructions better (use system prompt)
- You have fewer than ~20 examples (results will be unreliable)


In [ ]:
# Part 1 — Decision ladder demo
# The SAME task solved at four rungs of the ladder — compare outputs and cost

TASK = "User says: 'My shipment hasn't arrived after 10 days.' Reply as a support agent."

# Rung 1: Zero-shot prompt
r1 = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent.'},
        {'role': 'user',   'content': TASK}
    ]
)

# Rung 2: Few-shot prompt (2 examples baked into the system message)
FEW_SHOT_EXAMPLES = """
Example 1:
User: "My order from 2 weeks ago still hasn't shipped."
Agent: "I sincerely apologize for the delay. I've flagged your order #XXXX for priority review 
and you'll receive a tracking update within 24 hours. Thank you for your patience."

Example 2:
User: "The package shows delivered but I never received it."
Agent: "I'm sorry to hear that. I've initiated a trace with our carrier and will follow up 
within 2 business days with a resolution — either a replacement or a full refund."
"""
r2 = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[
        {'role': 'system', 'content': f'You are a helpful customer support agent.\n{FEW_SHOT_EXAMPLES}'},
        {'role': 'user',   'content': TASK}
    ]
)

print('--- Rung 1: Zero-shot ---')
print(r1.choices[0].message.content)
print('\n--- Rung 2: Few-shot ---')
print(r2.choices[0].message.content)
print('\nObservation: How do these differ in tone, length, and specificity?')


---
## 🔹 Part 2 — Preparing Training Data (JSONL)

Fine-tuning requires a **JSONL** file (JSON Lines — one JSON object per line). Each example is a conversation:

```json
{"messages": [
  {"role": "system",    "content": "You are a support agent."},
  {"role": "user",      "content": "My shipment is late."},
  {"role": "assistant", "content": "I sincerely apologize..."}
]}
```

**Rules of thumb for training data quality:**
- Minimum **10 examples** for the API to accept the job; **50+** for consistent results
- Every example must have the same system message you'll use at inference
- The assistant turn is what the model **learns to imitate** — make it high quality
- Avoid contradictory examples (model will learn an average)


In [ ]:
# ============================================================
# Part 2 — Build and validate a JSONL training file
# ============================================================
import json

SYSTEM_MSG = (
    'You are Alex, a friendly and empathetic customer support agent for ShipFast. '
    'Always acknowledge the customer\'s frustration, provide a clear next step, and close with a timeline. '
    'Keep responses under 60 words.'
)

# 20 labeled examples — this is your training set
EXAMPLES = [
    ('My shipment hasn\'t arrived after 10 days.',
     'I\'m sorry for the delay! I\'ve opened a trace on your shipment and escalated it to priority. You\'ll receive a tracking update or replacement decision within 24 hours. Thank you for your patience, and I sincerely apologize for the inconvenience.'),
    ('The package shows delivered but I never received it.',
     'That\'s frustrating — let me get this resolved. I\'ve filed a carrier investigation and will follow up within 2 business days with either a replacement or a full refund. Please check with neighbors or your building office in the meantime.'),
    ('I ordered the wrong size. Can I exchange it?',
     'Absolutely! Exchanges are easy. I\'ve sent a prepaid return label to your email — just drop it off at any carrier location within 14 days. Your new size will ship the same day we receive the return.'),
    ('My order was cancelled without my permission.',
     'I sincerely apologize — that should never happen. I\'ve reviewed the record and see a payment verification issue triggered the cancellation. I\'m reinstating your order now and waiving the shipping fee. You\'ll receive a confirmation email within the hour.'),
    ('Where is my order? It\'s been 3 days.',
     'Your order is currently in transit! Tracking shows it left the warehouse yesterday and is estimated to arrive within 2 business days. I\'ll send you a direct tracking link right now.'),
    ('I received the wrong item.',
     'Oh no — I\'m sorry about that! I\'ve flagged this as a pick-pack error and arranged an immediate replacement. A return label is on its way to your email. The correct item will ship today.'),
    ('Can I change my delivery address?',
     'If your order hasn\'t shipped yet, I can update the address right now — just reply with the new one. If it\'s already in transit, I\'ll contact the carrier to request a redirect, though this isn\'t guaranteed. What is the correct address?'),
    ('The item arrived damaged.',
     'I\'m truly sorry to see that. Could you send a photo to support@shipfast.com? Once I confirm the damage I\'ll ship a replacement at no charge or issue a full refund — your choice. This will be processed within one business day.'),
    ('I haven\'t received a confirmation email.',
     'Let me check on that. Your order was placed successfully — the confirmation may have landed in your spam folder. I\'m resending it now. If it doesn\'t arrive within 10 minutes please reach back out.'),
    ('How long does standard shipping take?',
     'Standard shipping takes 3–5 business days within the continental US. Expedited (1–2 days) and overnight options are available at checkout. International orders vary by destination — typically 7–14 business days.'),
    ('I want to return my purchase.',
     'No problem! Returns are accepted within 30 days of delivery. I\'ve generated a prepaid return label and emailed it to you. Once the item is received and inspected (1–3 business days), your refund will be issued to the original payment method.'),
    ('My coupon code isn\'t working.',
     'I\'d be happy to help. Could you share the code you\'re trying? Common issues are expired dates or category restrictions. If the code is valid and still failing, I\'ll apply the discount manually on my end right now.'),
    ('I was charged twice for my order.',
     'I sincerely apologize for the double charge — that\'s definitely our error. I\'ve initiated a refund for the duplicate amount. It should appear on your statement within 3–5 business days depending on your bank. Thank you for catching this.'),
    ('Can I cancel my order?',
     'If your order hasn\'t shipped I can cancel it immediately and issue a full refund. Let me check the status now. If it has already shipped, I\'ll arrange a free return the moment it arrives. Either way, you won\'t pay a cent.'),
    ('Do you ship internationally?',
     'Yes! We ship to 40+ countries. International shipping rates and delivery windows are shown at checkout based on your destination. Duties and customs fees are the buyer\'s responsibility and vary by country.'),
    ('My tracking hasn\'t updated in 5 days.',
     'Tracking gaps like this sometimes happen in transit, especially at customs or between carrier hubs. I\'ve flagged your shipment for a manual check with the carrier. I\'ll have an update for you within 48 hours. Thank you for your patience.'),
    ('I need my order by Friday — is that possible?',
     'Let me check! If you order before 2 PM today, overnight shipping will deliver by Thursday. Expedited (2-day) would arrive Friday if placed within the next hour. Want me to upgrade your shipping? I can waive the fee given the urgency.'),
    ('The size chart on the website seems wrong.',
     'Thank you for flagging that — I\'ll pass it to our product team right away. For your current order, I recommend sizing up if you\'re between sizes. If it doesn\'t fit, free exchanges are available within 30 days.'),
    ('I accidentally ordered twice.',
     'No worries — I can cancel the duplicate order right now. Which order number would you like to keep? Once you confirm, I\'ll cancel the other immediately and issue a full refund within 3–5 business days.'),
    ('Your website was down and I couldn\'t track my order.',
     'I apologize for any disruption — we had a brief maintenance window this morning. Everything is back online now. Your order is currently in transit and on schedule to arrive within the original delivery window. Here is your tracking link: [tracking_url].'),
]

def make_example(user_msg: str, assistant_msg: str) -> dict:
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_MSG},
            {'role': 'user',      'content': user_msg},
            {'role': 'assistant', 'content': assistant_msg},
        ]
    }

training_data = [make_example(u, a) for u, a in EXAMPLES]

# Save to JSONL
TRAIN_FILE = 'shipfast_support_train.jsonl'
with open(TRAIN_FILE, 'w') as f:
    for row in training_data:
        f.write(json.dumps(row) + '\n')

print(f'Saved {len(training_data)} examples to {TRAIN_FILE}')
print('\nFirst example preview:')
print(json.dumps(training_data[0], indent=2))


In [ ]:
# Validate the JSONL before uploading
import tiktoken

def count_tokens(messages: list, model: str = 'gpt-4.1-mini') -> int:
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding('cl100k_base')
    total = 0
    for m in messages:
        total += 4  # role + content overhead
        total += len(enc.encode(m.get('content', '')))
    return total + 2  # priming tokens

token_counts = [count_tokens(ex['messages']) for ex in training_data]
total_tokens = sum(token_counts)

print(f'Examples:      {len(training_data)}')
print(f'Total tokens:  {total_tokens:,}')
print(f'Avg per example: {total_tokens // len(training_data)}')
print(f'Max per example: {max(token_counts)}')
print(f'\nEstimated fine-tune cost (1 epoch): ~${total_tokens * 3 / 1_000_000:.4f}')
print('(gpt-4.1-mini fine-tune: $3.00 / 1M training tokens at 1 epoch)')

---
## 🔹 Part 3 — Submitting the Fine-Tune Job

The OpenAI fine-tuning API has three steps:
1. **Upload** the JSONL file → get a `file_id`
2. **Create** the fine-tune job → get a `job_id`
3. **Poll** until the job finishes → get a `fine_tuned_model` name

Set `SUBMIT_JOB = True` to actually run the job (~$0.30, ~15 min). Set `SUBMIT_JOB = False` to walk through the API calls without billing.


In [ ]:
# ============================================================
# Part 3 — Fine-Tune Job: Upload → Create → Poll
# ============================================================
import time

SUBMIT_JOB = False   # ← set True to actually bill and run
FT_BASE_MODEL = 'gpt-4.1-mini-2025-04-14'   # supported fine-tune target as of 2026

if SUBMIT_JOB:
    # Step 1: Upload training file
    print('Step 1: Uploading training file...')
    with open(TRAIN_FILE, 'rb') as f:
        file_resp = client.files.create(file=f, purpose='fine-tune')
    file_id = file_resp.id
    print(f'  File uploaded: {file_id}')

    # Step 2: Create the fine-tune job
    print('\nStep 2: Creating fine-tune job...')
    ft_job = client.fine_tuning.jobs.create(
        training_file=file_id,
        model=FT_BASE_MODEL,
        hyperparameters={'n_epochs': 3},
    )
    job_id = ft_job.id
    print(f'  Job created: {job_id}')
    print(f'  Status: {ft_job.status}')

    # Step 3: Poll until done
    print('\nStep 3: Polling for completion (this takes ~15 min)...')
    while True:
        status = client.fine_tuning.jobs.retrieve(job_id)
        print(f'  [{time.strftime("%H:%M:%S")}] Status: {status.status}')
        if status.status in ('succeeded', 'failed', 'cancelled'):
            break
        time.sleep(60)

    if status.status == 'succeeded':
        FINE_TUNED_MODEL = status.fine_tuned_model
        print(f'\n✅ Fine-tuned model: {FINE_TUNED_MODEL}')
    else:
        print(f'\n❌ Job ended with status: {status.status}')
        FINE_TUNED_MODEL = None

else:
    # --- DRY RUN: show the API calls without billing ---
    print('DRY RUN (SUBMIT_JOB = False). These are the calls that would execute:\n')
    print('# Step 1: Upload')
    print("  client.files.create(file=open('shipfast_support_train.jsonl','rb'), purpose='fine-tune')")
    print("  # → {'id': 'file-abc123', 'status': 'processed'}")
    print('\n# Step 2: Create job')
    print(f"  client.fine_tuning.jobs.create(training_file='file-abc123', model='{FT_BASE_MODEL}', hyperparameters={{'n_epochs': 3}})")
    print("  # → {'id': 'ftjob-xyz789', 'status': 'queued'}")
    print('\n# Step 3: Poll')
    print("  client.fine_tuning.jobs.retrieve('ftjob-xyz789')")
    print("  # after ~15 min → {'status': 'succeeded', 'fine_tuned_model': 'ft:gpt-4.1-mini-2025-04-14:org:shipfast:abc123'}")
    FINE_TUNED_MODEL = 'ft:gpt-4.1-mini-2025-04-14:example:shipfast:abc123'  # placeholder for demo

---
## 🔹 Part 4 — Evaluate: Base vs Fine-Tuned

After fine-tuning, compare the **base model** (with the full system prompt) against the **fine-tuned model** (which has the style baked into its weights — you can use a shorter system prompt).

You should observe:
- **Shorter prompts** with the same or better quality
- **More consistent brand voice** (Alex persona, empathetic tone, action + timeline)
- **Lower token usage** per call → lower cost at scale


In [ ]:
# ============================================================
# Part 4 — Side-by-Side Evaluation
# ============================================================

HELD_OUT = [
    "I've been waiting 12 days and still nothing.",
    "I need to change the delivery address immediately.",
    "Got a broken product. This is unacceptable.",
]

def call_base(user_msg: str) -> str:
    r = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_MSG},
            {'role': 'user',   'content': user_msg},
        ]
    )
    return r.choices[0].message.content

def call_ft(user_msg: str, ft_model: str) -> str:
    # Fine-tuned model — shorter system prompt since style is baked in
    r = client.chat.completions.create(
        model=ft_model,
        messages=[
            {'role': 'system', 'content': 'You are Alex, a ShipFast support agent.'},
            {'role': 'user',   'content': user_msg},
        ]
    )
    return r.choices[0].message.content

print('=== Held-Out Evaluation ===')
for msg in HELD_OUT:
    print(f'\n--- User: {msg}')
    base_reply = call_base(msg)
    print(f'Base model:  {base_reply}')

    if FINE_TUNED_MODEL and SUBMIT_JOB:
        ft_reply = call_ft(msg, FINE_TUNED_MODEL)
        print(f'Fine-tuned:  {ft_reply}')
    else:
        print('Fine-tuned:  [Set SUBMIT_JOB=True and complete Part 3 to compare]')


---
## 🔹 Part 5 — Deploy with Gradio

Gradio turns any Python function into a shareable web UI with one line of code. It is the fastest way to demo a fine-tuned model to stakeholders or evaluate it interactively.


In [ ]:
# ============================================================
# Part 5 — Gradio Demo
# ============================================================
!pip install -q gradio
import gradio as gr

# Use whichever model is available
DEMO_MODEL = FINE_TUNED_MODEL if (FINE_TUNED_MODEL and SUBMIT_JOB) else DEFAULT_MINI_MODEL
DEMO_SYSTEM = (
    'You are Alex, a ShipFast support agent.'
    if (FINE_TUNED_MODEL and SUBMIT_JOB)
    else SYSTEM_MSG
)

def support_chat(user_message: str) -> str:
    r = client.chat.completions.create(
        model=DEMO_MODEL,
        messages=[
            {'role': 'system', 'content': DEMO_SYSTEM},
            {'role': 'user',   'content': user_message},
        ]
    )
    return r.choices[0].message.content

model_label = f'Fine-tuned: {DEMO_MODEL}' if SUBMIT_JOB else f'Base: {DEMO_MODEL}'

demo = gr.Interface(
    fn=support_chat,
    inputs=gr.Textbox(label='Customer message', placeholder='e.g. My shipment is 5 days late.', lines=3),
    outputs=gr.Textbox(label='Alex (ShipFast Support)', lines=4),
    title='ShipFast Support Bot',
    description=f'Powered by: {model_label}',
    examples=[
        ['My order arrived damaged — what do I do?'],
        ['Can I still cancel? I ordered an hour ago.'],
        ['I got charged twice for the same item.'],
    ],
)

demo.launch(share=True)   # share=True generates a public Gradio link


---
## ✋ Hands-On — Build Your Own Training Dataset

**Task:** Create a JSONL fine-tuning dataset for a **different domain** of your choice. Requirements:
- Minimum **20 examples**
- A consistent **system message** that defines the persona
- Realistic user messages + high-quality assistant responses
- Validate the token count and estimate the fine-tune cost

**Domain ideas:** legal assistant, medical intake bot, restaurant reservation agent, code review bot, HR onboarding chatbot

Fill in the `-----` placeholders below.


In [ ]:
# ✋ Hands-On: Your Fine-Tuning Dataset

MY_SYSTEM_MSG = '-----'   # Define your persona (1–2 sentences)

MY_EXAMPLES = [
    # ('user message', 'ideal assistant response'),
    ('-----', '-----'),   # example 1
    ('-----', '-----'),   # example 2
    # ... add at least 18 more
]

# Validate
if len(MY_EXAMPLES) >= 10 and MY_SYSTEM_MSG.strip() != '-----':
    my_training_data = [make_example(u, a) for u, a in MY_EXAMPLES]
    my_token_counts  = [count_tokens(ex['messages']) for ex in my_training_data]
    my_total         = sum(my_token_counts)
    print(f'Examples:      {len(my_training_data)}')
    print(f'Total tokens:  {my_total:,}')
    print(f'Estimated cost: ~${my_total * 3 / 1_000_000:.4f} for 1 epoch of gpt-4.1-mini')

    MY_TRAIN_FILE = 'my_finetune_train.jsonl'
    with open(MY_TRAIN_FILE, 'w') as f:
        for row in my_training_data:
            f.write(json.dumps(row) + '\n')
    print(f'Saved to {MY_TRAIN_FILE}')
else:
    print('Fill in MY_SYSTEM_MSG and add at least 10 MY_EXAMPLES to validate.')

---
## 📋 Assignment Checklist

Your submission for M13 should include:

- [ ] **Dataset JSONL** — at least 20 examples in your chosen domain
- [ ] **Token count + cost estimate** — show your validation cell output
- [ ] **Base model output** — 3 held-out responses from the base model
- [ ] **Fine-tuned model output** (if budget allows) — same 3 prompts on your fine-tuned model
- [ ] **Gradio demo link** — shareable Gradio URL (the `share=True` link)
- [ ] **Analysis paragraph** — when would fine-tuning be justified for your domain vs just using RAG or prompt engineering?


<hr>
<h2 style="color:#2e7d32;">🎉 Lab Complete</h2>

You walked the full fine-tuning pipeline:

- ✅ **Decision framework** — when fine-tuning beats prompt/RAG/tools
- ✅ **JSONL data prep** — 20 labeled examples, validation, token costing
- ✅ **Fine-tune job API** — upload → create → poll → retrieve model
- ✅ **Side-by-side eval** — base vs fine-tuned on held-out prompts
- ✅ **Gradio deployment** — live shareable demo in 10 lines

The expensive part is always the data — high-quality labeled examples are the bottleneck, not the training compute.
